# Additional Local-Search Operators

This notebook evaluates whether additional local-search operators improve the solution after four applications of `move_plateau`.

It produces

1. a comparison of relative solution quality
2. a comparison of runtime

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [2]:
GRAPH_ORDER = ["powerlaw", "er"]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

BASE_PIPELINE = (
    "move_plateau,move_plateau,move_plateau,move_plateau"
)

ADDITIONAL_OPERATORS = [
    "merge_first",
    "merge_best",
    "bridge_split",
    "split_min_cut",
]

PIPELINE_ORDER = [
    BASE_PIPELINE,
    *[
        f"{BASE_PIPELINE},{operator}"
        for operator in ADDITIONAL_OPERATORS
    ],
]

PIPELINE_LABELS = {
    BASE_PIPELINE: "move_plateau",
    **{
        f"{BASE_PIPELINE},{operator}": operator
        for operator in ADDITIONAL_OPERATORS
    },
}

RESULTS_DIR = Path("../results/experiment2/extra_operator")

RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"
STEP_RESULTS_FILE = RESULTS_DIR / "step_results.csv"

## Load and verify experiment data

Before the analysis, the notebook lists the contained pipelines, start partitions, zero-gain factors, and number of runs. This provides a compact check that the intended experiment results were loaded.

In [3]:
step_all = pd.read_csv(STEP_RESULTS_FILE)

required_columns = {
    "experiment",
    "pipeline",
    "start_partition",
    "zero_gain_factor",
    "run",
    "graph_type",
    "size_class",
    "regime",
    "dataset",
    "instance",
    "step_index",
    "step_name",
    "score_after",
    "runtime",
}

missing_columns = required_columns.difference(step_all.columns)

if missing_columns:
    raise ValueError("Missing required columns: " + ", ".join(sorted(missing_columns)))

experiment_check = (
    step_all
    .groupby(
        [
            "pipeline",
            "start_partition",
            "zero_gain_factor",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        num_runs=("run", "nunique"),
        num_instances=("instance", "nunique"),
        num_steps=("step_index", "nunique"),
        num_rows=("run", "size"),
    )
    .sort_values(
        [
            "pipeline",
            "start_partition",
            "zero_gain_factor",
        ],
        na_position="first",
    )
    .reset_index(drop=True)
)

print(f"Loaded {len(step_all):,} rows from {STEP_RESULTS_FILE}")

experiment_check

Loaded 480,000 rows from ../results/experiment2/extra_operator/step_results.csv


,pipeline,start_partition,zero_gain_factor,num_runs,num_instances,num_steps,num_rows
0,"move_plateau,move_plateau,move_plateau,move_pl...",maximum_matching,4,10,2000,4,80000
1,"move_plateau,move_plateau,move_plateau,move_pl...",maximum_matching,4,10,2000,5,100000
2,"move_plateau,move_plateau,move_plateau,move_pl...",maximum_matching,4,10,2000,5,100000
3,"move_plateau,move_plateau,move_plateau,move_pl...",maximum_matching,4,10,2000,5,100000
4,"move_plateau,move_plateau,move_plateau,move_pl...",maximum_matching,4,10,2000,5,100000


## Prepare extra-operator experiments

In [4]:
step_all["dataset_group"] = (
        step_all["size_class"].astype(str)
        + " "
        + step_all["regime"].astype(str)
)

step_all["graph_type"] = pd.Categorical(
    step_all["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

step_all["dataset_group"] = pd.Categorical(
    step_all["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

operator_step_results = step_all[
    step_all["pipeline"].isin(PIPELINE_ORDER)
    & (step_all["start_partition"] == "maximum_matching")
    ].copy()

if operator_step_results.empty:
    raise ValueError("No rows found for the expected operator pipelines.")

operator_step_results["pipeline"] = pd.Categorical(
    operator_step_results["pipeline"],
    categories=PIPELINE_ORDER,
    ordered=True,
)

In [5]:
execution_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "pipeline",
    "run",
]

final_steps = (
    operator_step_results
    .sort_values(execution_keys + ["step_index"])
    .groupby(
        execution_keys,
        observed=True,
        as_index=False,
    )
    .tail(1)
    .reset_index(drop=True)
)

In [6]:
best_run_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "pipeline",
]

best_operator_steps = (
    final_steps
    .sort_values(
        [
            "score_after",
            "runtime",
            "run",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .groupby(
        best_run_keys,
        observed=True,
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

## Relative solution quality

For every instance and pipeline, only the run with the highest final solution quality is retained.

The pipeline consisting only of four applications of `move_plateau` serves as the reference. The relative solution quality of an extended pipeline is therefore defined as

$
\frac{\text{solution quality after four applications of } \texttt{move\_plateau}}
     {\text{solution quality of the extended pipeline}}.
$

A value of $1.0$ indicates that the additional operator does not change the final solution quality.
Values smaller than $1.0$ indicate that the additional operator improves the final solution.

The reported values are averaged separately for every graph model, graph size, and density regime.

In [7]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = (
    best_operator_steps
    .pivot(
        index=instance_keys,
        columns="pipeline",
        values="score_after",
    )
)

density_table = density_table[PIPELINE_ORDER]

if density_table.isna().any().any():
    incomplete_instances = density_table[density_table.isna().any(axis=1)]

    raise ValueError(
        f"{len(incomplete_instances)} instances do not contain results for every pipeline."
    )

In [8]:
reference_score = density_table[PIPELINE_ORDER[0]]

relative_to_reference = density_table.rdiv(reference_score, axis=0)
relative_to_reference.columns.name = "pipeline"

relative_quality_summary = (
    relative_to_reference
    .groupby(level=["graph_type", "dataset_group"])
    .mean()
    .stack()
    .rename("mean_relative_quality_to_reference")
    .reset_index()
)

relative_quality_summary["operator"] = (
    relative_quality_summary["pipeline"]
    .astype(str)
    .map(PIPELINE_LABELS)
)

relative_quality_summary = (
    relative_quality_summary[
        relative_quality_summary["operator"]
        .isin(ADDITIONAL_OPERATORS)
    ]
        .drop(columns="pipeline")
)

relative_quality_summary["operator"] = pd.Categorical(
    relative_quality_summary["operator"],
    categories=ADDITIONAL_OPERATORS,
    ordered=True,
)

relative_quality_summary = (
    relative_quality_summary
    .sort_values(
        [
            "graph_type",
            "dataset_group",
            "operator",
        ]
    )
    .reset_index(drop=True)
)

relative_quality_summary = relative_quality_summary[
    [
        "graph_type",
        "dataset_group",
        "operator",
        "mean_relative_quality_to_reference",
    ]
]

relative_quality_summary

,graph_type,dataset_group,operator,mean_relative_quality_to_reference
0,powerlaw,small sparse,merge_first,0.999963
1,powerlaw,small sparse,merge_best,0.999963
2,powerlaw,small sparse,bridge_split,0.999958
3,powerlaw,small sparse,split_min_cut,1.000000
4,powerlaw,small dense,merge_first,0.999973
5,powerlaw,small dense,merge_best,0.999973
6,powerlaw,small dense,bridge_split,1.000000
7,powerlaw,small dense,split_min_cut,1.000000
8,powerlaw,large sparse,merge_first,1.000000
9,powerlaw,large sparse,merge_best,1.000000


## Operator activity and runtime

For the activity analysis, only the best-performing run of each instance and pipeline is considered.

The following activity statistics are reported:

- **Affected instances:** number of instances on which the additional operator performs at least one move in the selected best run.
- **Affected instance rate:** fraction of instances on which the additional operator performs at least one move in the selected best run.
- **Total moves:** total number of accepted moves performed by the additional operator across the selected best runs.

For the runtime analysis, all randomized runs are considered. For each instance and additional operator, the operator runtimes of all ten runs are summed. These total runtimes are then averaged over all instances in the corresponding graph and dataset group.

In [9]:
additional_operator_steps_best_runs = best_operator_steps[
    best_operator_steps["step_name"].isin(ADDITIONAL_OPERATORS)
].copy()

operator_activity_summary = (
    additional_operator_steps_best_runs
    .groupby(
        [
            "graph_type",
            "dataset_group",
            "step_name",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        affected_instances=(
            "num_moves",
            lambda s: (s > 0).sum(),
        ),
        affected_instance_percent=(
            "num_moves",
            lambda s: 100 * (s > 0).mean(),
        ),
        total_moves=(
            "num_moves",
            "sum",
        ),
    )
    .rename(
        columns={
            "step_name": "operator",
        }
    )
)

In [10]:
additional_operator_steps_all_runs = final_steps[
    final_steps["step_name"].isin(ADDITIONAL_OPERATORS)
].copy()

runtime_per_instance = (
    additional_operator_steps_all_runs
    .groupby(
        [
            "graph_type",
            "dataset_group",
            "dataset",
            "instance",
            "step_name",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        total_operator_runtime=(
            "runtime",
            "sum",
        ),
        num_runs=(
            "run",
            "nunique",
        ),
    )
)

operator_runtime_summary = (
    runtime_per_instance
    .groupby(
        [
            "graph_type",
            "dataset_group",
            "step_name",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        mean_total_operator_runtime=(
            "total_operator_runtime",
            "mean",
        )
    )
    .rename(
        columns={
            "step_name": "operator",
        }
    )
)

In [11]:
operator_summary = (
    operator_activity_summary
    .merge(
        operator_runtime_summary,
        on=[
            "graph_type",
            "dataset_group",
            "operator",
        ],
        how="left",
        validate="one_to_one",
    )
)

operator_summary["operator"] = pd.Categorical(
    operator_summary["operator"],
    categories=ADDITIONAL_OPERATORS,
    ordered=True,
)

operator_summary = (
    operator_summary
    .sort_values(
        [
            "graph_type",
            "dataset_group",
            "operator",
        ]
    )
    .reset_index(drop=True)
)

operator_summary

,graph_type,dataset_group,operator,affected_instances,affected_instance_percent,total_moves,mean_total_operator_runtime
0,powerlaw,small sparse,merge_first,2,0.8,2,0.101804
1,powerlaw,small sparse,merge_best,2,0.8,2,0.098959
2,powerlaw,small sparse,bridge_split,3,1.2,3,0.109508
3,powerlaw,small sparse,split_min_cut,0,0.0,0,0.116094
4,powerlaw,small dense,merge_first,4,1.6,4,0.135439
5,powerlaw,small dense,merge_best,4,1.6,4,0.131186
6,powerlaw,small dense,bridge_split,0,0.0,0,0.130072
7,powerlaw,small dense,split_min_cut,0,0.0,0,0.142094
8,powerlaw,large sparse,merge_first,0,0.0,0,0.826063
9,powerlaw,large sparse,merge_best,0,0.0,0,0.848652


## LaTeX helper functions

In [12]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor

def latex_operator(operator: str) -> str:
    return r"\texttt{" + operator.replace("_", r"\_") + "}"

def format_number(value: float, decimals: int) -> str:
    return f"{value:.{decimals}f}"

def format_percent(value: float, decimals: int = 1) -> str:
    return rf"{truncate_number(value, decimals):.{decimals}f}\,\%"

## Build quality LaTeX table

In [13]:
def make_quality_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    graph_df = df[df["graph_type"] == graph_type].copy()

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lp{4.0cm}r}",
        r"\toprule",
        (
            r"Datensatz & Operator "
            r"& \shortstack{Mittlere relative\\Lösungsqualität} \\"
        ),
        r"\midrule",
    ]

    for dataset_index, dataset in enumerate(DATASET_ORDER):
        part = graph_df[graph_df["dataset_group"] == dataset].copy()

        if part.empty:
            continue

        part["operator"] = pd.Categorical(
            part["operator"],
            categories=ADDITIONAL_OPERATORS,
            ordered=True,
        )

        part = part.sort_values("operator")

        best_quality = part["mean_relative_quality_to_reference"].min()

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset}}}"
                if row_index == 0
                else ""
            )

            quality = format_number(row.mean_relative_quality_to_reference,6,)

            if (row.mean_relative_quality_to_reference < 1.0 and
                    np.isclose(row.mean_relative_quality_to_reference, best_quality)):
                quality = rf"\textbf{{{quality}}}"

            lines.append(
                f"{dataset_cell} "
                f"& {latex_operator(str(row.operator))} "
                f"& {quality} "
                r"\\"
            )

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\cmidrule(l){1-3}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [14]:
powerlaw_quality_latex = make_quality_latex_table(
    relative_quality_summary,
    graph_type="powerlaw",
    caption=(
        "Mittlere relative Lösungsqualität der zusätzlichen Local-Search-Operatoren auf Powerlaw-Instanzen gegenüber der Referenzpipeline aus vier Anwendungen von \\texttt{move\\_plateau}. Werte kleiner als 1 zeigen eine Verbesserung gegenüber der Referenz."
    ),
    label="tab:additional_operator_quality_powerlaw",
)

print(powerlaw_quality_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität der zusätzlichen Local-Search-Operatoren auf Powerlaw-Instanzen gegenüber der Referenzpipeline aus vier Anwendungen von \texttt{move\_plateau}. Werte kleiner als 1 zeigen eine Verbesserung gegenüber der Referenz.}
\label{tab:additional_operator_quality_powerlaw}
\begin{tabular}{lp{4.0cm}r}
\toprule
Datensatz & Operator & \shortstack{Mittlere relative\\Lösungsqualität} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & \textbf{0.999963} \\
 & \texttt{merge\_best} & \textbf{0.999963} \\
 & \texttt{bridge\_split} & \textbf{0.999958} \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & \textbf{0.999973} \\
 & \texttt{merge\_best} & \textbf{0.999973} \\
 & \texttt{bridge\_split} & 1.000000 \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{large sparse} & \texttt{merge\_first} & 1.000000 \\
 & \texttt{merge\_best

In [15]:
er_quality_latex = make_quality_latex_table(
    relative_quality_summary,
    graph_type="er",
    caption=(
        "Mittlere relative Lösungsqualität der zusätzlichen Local-Search-Operatoren auf Erdős--Rényi-Instanzen gegenüber der Referenzpipeline aus vier Anwendungen von \\texttt{move\\_plateau}. Werte kleiner als 1 zeigen eine Verbesserung gegenüber der Referenz."
    ),
    label="tab:additional_operator_quality_er",
)

print(er_quality_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität der zusätzlichen Local-Search-Operatoren auf Erdős--Rényi-Instanzen gegenüber der Referenzpipeline aus vier Anwendungen von \texttt{move\_plateau}. Werte kleiner als 1 zeigen eine Verbesserung gegenüber der Referenz.}
\label{tab:additional_operator_quality_er}
\begin{tabular}{lp{4.0cm}r}
\toprule
Datensatz & Operator & \shortstack{Mittlere relative\\Lösungsqualität} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & 1.000000 \\
 & \texttt{merge\_best} & 1.000000 \\
 & \texttt{bridge\_split} & \textbf{0.999953} \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & \textbf{0.999977} \\
 & \texttt{merge\_best} & \textbf{0.999977} \\
 & \texttt{bridge\_split} & 1.000000 \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{large sparse} & \texttt{merge\_first} & 1.000000 \\
 & \texttt{merge\_best} & 1.000000 \\
 & \

## Build operator activity LaTeX tables

In [19]:
def make_activity_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    graph_df = df[df["graph_type"] == graph_type].copy()

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lp{3.4cm}rrrr}",
        r"\toprule",
        (
            r"Datensatz & Operator "
            r"& \shortstack{Betroffene\\Instanzen} "
            r"& Anteil "
            r"& Moves "
            r"& \shortstack{Gesamt-\\laufzeit (s)} \\"
        ),
        r"\midrule",
    ]

    for dataset_index, dataset in enumerate(DATASET_ORDER):
        part = graph_df[graph_df["dataset_group"] == dataset].copy()

        if part.empty:
            continue

        part["operator"] = pd.Categorical(
            part["operator"],
            categories=ADDITIONAL_OPERATORS,
            ordered=True,
        )

        part = part.sort_values("operator")

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset}}}"
                if row_index == 0
                else ""
            )

            lines.append(
                f"{dataset_cell} "
                f"& {latex_operator(str(row.operator))} "
                f"& {int(row.affected_instances)} "
                f"& {format_percent(row.affected_instance_percent, 1)} "
                f"& {int(row.total_moves)} "
                f"& {format_number(row.mean_total_operator_runtime, 4)} "
                r"\\"
            )

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\cmidrule(l){1-6}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [20]:
powerlaw_activity_latex = make_activity_latex_table(
    operator_summary,
    graph_type="powerlaw",
    caption=(
        "Aktivität und mittlere Laufzeit der zusätzlichen Local-Search-Operatoren in den jeweils besten Runs auf Powerlaw-Instanzen. Als betroffen gilt eine Instanz, wenn der zusätzliche Operator mindestens einen Zug ausführt."
    ),
    label="tab:additional_operator_activity_powerlaw",
)

print(powerlaw_activity_latex)

\begin{table}[t]
\centering
\caption{Aktivität und mittlere Laufzeit der zusätzlichen Local-Search-Operatoren in den jeweils besten Runs auf Powerlaw-Instanzen. Als betroffen gilt eine Instanz, wenn der zusätzliche Operator mindestens einen Zug ausführt.}
\label{tab:additional_operator_activity_powerlaw}
\begin{tabular}{lp{3.4cm}rrrr}
\toprule
Datensatz & Operator & \shortstack{Betroffene\\Instanzen} & Anteil & Moves & \shortstack{Gesamt-\\laufzeit (s)} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & 2 & 0.8\,\% & 2 & 0.1018 \\
 & \texttt{merge\_best} & 2 & 0.8\,\% & 2 & 0.0990 \\
 & \texttt{bridge\_split} & 3 & 1.2\,\% & 3 & 0.1095 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1161 \\
\cmidrule(l){1-6}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & 4 & 1.6\,\% & 4 & 0.1354 \\
 & \texttt{merge\_best} & 4 & 1.6\,\% & 4 & 0.1312 \\
 & \texttt{bridge\_split} & 0 & 0.0\,\% & 0 & 0.1301 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1421 \\
\cmidrule

In [21]:
er_activity_latex = make_activity_latex_table(
    operator_summary,
    graph_type="er",
    caption=(
        "Aktivität und mittlere Laufzeit der zusätzlichen Local-Search-Operatoren in den jeweils besten Runs auf Erdős-Rényi-Instanzen. Als betroffen gilt eine Instanz, wenn der zusätzliche Operator mindestens einen Zug ausführt."
    ),
    label="tab:additional_operator_activity_er",
)

print(er_activity_latex)

\begin{table}[t]
\centering
\caption{Aktivität und mittlere Laufzeit der zusätzlichen Local-Search-Operatoren in den jeweils besten Runs auf Erdős-Rényi-Instanzen. Als betroffen gilt eine Instanz, wenn der zusätzliche Operator mindestens einen Zug ausführt.}
\label{tab:additional_operator_activity_er}
\begin{tabular}{lp{3.4cm}rrrr}
\toprule
Datensatz & Operator & \shortstack{Betroffene\\Instanzen} & Anteil & Moves & \shortstack{Gesamt-\\laufzeit (s)} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & 0 & 0.0\,\% & 0 & 0.1085 \\
 & \texttt{merge\_best} & 1 & 0.4\,\% & 1 & 0.1039 \\
 & \texttt{bridge\_split} & 1 & 0.4\,\% & 1 & 0.0983 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1019 \\
\cmidrule(l){1-6}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & 4 & 1.6\,\% & 4 & 0.1515 \\
 & \texttt{merge\_best} & 4 & 1.6\,\% & 4 & 0.1447 \\
 & \texttt{bridge\_split} & 0 & 0.0\,\% & 0 & 0.1210 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1294 \\
\cmidrule(l)